# Late Delivery Risk Prediction for E-Commerce Supply Chain

**Capstone – Main notebook**

- **Track:** Supervised Learning — Binary Classification  
- **Dataset:** E-commerce supply chain (order, shipping, customer, product)  
- **Target:** `Late_delivery_risk` (0 = on time, 1 = late)  
- **Goals:** Recall ≥ 75%, F1 ≥ 70% on late class; interpretable feature importance.

---

## 1. Setup and load data

**Colab:** Upload CSV via **Files** (📁) → Upload, then set `data_path` in the load cell (e.g. `Path("/content/your_file.csv")`).  
**Local:** Put CSV in `data/` (e.g. `data/supply_chain.csv`).

In [1]:
# Run once if you get ModuleNotFoundError; then restart kernel and run again.
%pip install pandas numpy matplotlib seaborn scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import libraries.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load data.
data_path = "../data/data.csv"

df = pd.read_csv(data_path, encoding='latin1')
print(f"Loaded: {data_path} -> shape {df.shape}")
df.shape

Loaded: ../data/data.csv -> shape (180519, 53)


(180519, 53)

### Proposal column check

Target: `Late_delivery_risk`. Key features: shipping days (real/scheduled), Shipping Mode, Order Region, Market, Customer Segment, Category Name, etc.

In [3]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,02-03-18 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [7]:
duplicates = df[df.duplicated(subset=['Order Customer Id'], keep=False)]
print(f"Number of duplicate rows based on Order Customer Id: {len(duplicates)}")
pd.set_option('display.max_columns', None)
display(duplicates.sort_values(by='Order Id').head(10))

Number of duplicate rows based on Order Customer Id: 172084


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
33833,CASH,2,4,88.790001,239.979996,Advance shipping,0,43,Camping & Hiking,Hickory,EE. UU.,XXXXXXXXX,Mary,11599,Malone,XXXXXXXXX,Consumer,NC,8708 Indian Horse Highway,28601.0,7,Fan Shop,35.776661,-81.362625,LATAM,Mexico City,México,11599,01-01-15 00:00,1,957,60.00,0.20,1,299.980011,0.37,1,299.980011,239.979996,88.790001,Central America,Distrito Federal,CLOSED,NaN,957,43,NaN,http://images.acmesports.sports/Diamondback+Wo...,Diamondback Women's Serene Classic Comfort Bi,299.980011,0,01-03-15 00:00,Standard Class
87884,PAYMENT,3,4,36.470001,107.889999,Advance shipping,0,18,Men's Footwear,Chicago,EE. UU.,XXXXXXXXX,David,256,Rodriguez,XXXXXXXXX,Consumer,IL,7605 Tawny Horse Falls,60625.0,4,Apparel,41.832722,-87.980484,LATAM,Dos Quebradas,Colombia,256,01-01-15 00:21,2,403,22.10,0.17,4,129.990005,0.34,1,129.990005,107.889999,36.470001,South America,Risaralda,PENDING_PAYMENT,NaN,403,18,NaN,http://images.acmesports.sports/Nike+Men%27s+C...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,0,01-04-15 00:21,Standard Class
77011,PAYMENT,3,4,91.180000,193.990005,Advance shipping,0,48,Water Sports,Chicago,EE. UU.,XXXXXXXXX,David,256,Rodriguez,XXXXXXXXX,Consumer,IL,7605 Tawny Horse Falls,60625.0,7,Fan Shop,41.832722,-87.980484,LATAM,Dos Quebradas,Colombia,256,01-01-15 00:21,2,1073,6.00,0.03,2,199.990005,0.47,1,199.990005,193.990005,91.180000,South America,Risaralda,PENDING_PAYMENT,NaN,1073,48,NaN,http://images.acmesports.sports/Pelican+Sunstr...,Pelican Sunstream 100 Kayak,199.990005,0,01-04-15 00:21,Standard Class
109322,PAYMENT,3,4,68.250000,227.500000,Advance shipping,0,24,Women's Apparel,Chicago,EE. UU.,XXXXXXXXX,David,256,Rodriguez,XXXXXXXXX,Consumer,IL,7605 Tawny Horse Falls,60625.0,5,Golf,41.832722,-87.980484,LATAM,Dos Quebradas,Colombia,256,01-01-15 00:21,2,502,22.50,0.09,3,50.000000,0.30,5,250.000000,227.500000,68.250000,South America,Risaralda,PENDING_PAYMENT,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+D...,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,01-04-15 00:21,Standard Class
63764,CASH,5,4,4.100000,40.980000,Late delivery,1,40,Accessories,San Antonio,EE. UU.,XXXXXXXXX,Brian,8827,Wilson,XXXXXXXXX,Home Office,TX,8396 High Corners,78240.0,6,Outdoors,29.520010,-98.637413,LATAM,Dos Quebradas,Colombia,8827,01-01-15 01:03,4,897,9.00,0.18,5,24.990000,0.10,2,49.980000,40.980000,4.100000,South America,Risaralda,CLOSED,NaN,897,40,NaN,http://images.acmesports.sports/Team+Golf+New+...,Team Golf New England Patriots Putter Grip,24.990000,0,01-06-15 01:03,Standard Class
95938,CASH,5,4,60.270000,123.000000,Late delivery,1,24,Women's Apparel,San Antonio,EE. UU.,XXXXXXXXX,Brian,8827,Wilson,XXXXXXXXX,Home Office,TX,8396 High Corners,78240.0,5,Golf,29.520010,-98.637413,LATAM,Dos Quebradas,Colombia,8827,01-01-15 01:03,4,502,27.00,0.18,7,50.000000,0.49,3,150.000000,123.000000,60.270000,South America,Risaralda,CLOSED,NaN,502,24,NaN,http://images.acmesports.sports/Nike+Men%27s+D...,Nike Men's Dri-FIT Victory Golf Polo,50.000000,0,01-06-15 01:03,Standard Class
114915,CASH,5,4,33.590000,159.940002,Late delivery,1,46,Indoor/Outdoor Games,San Antonio,EE. UU.,XXXXXXXXX,Brian,8827,Wilson,

---

## 2. Data cleaning

Drop rows with missing target, remove duplicates, fix types. Document assumptions.

In [ ]:
# Your cleaning steps here.
# work = df.dropna(subset=[TARGET]).drop_duplicates()  # example
work = df.copy()  # replace with cleaned df
work.shape

---

## 3. EDA

Distributions, late vs on-time by shipping mode / region / segment.

In [ ]:
# Your EDA and plots here.

---

## 4. Feature engineering

Encode categoricals, derive delay gap (real − scheduled days), build feature matrix X and target y. Do not use Delivery Status as a feature (leakage).

In [ ]:
# Build X, y. Example: delay_gap = real_days - scheduled_days; one-hot or ordinal encode categoricals.
# X = ...
# y = work[TARGET]
# train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

---

## 5. Baseline — Logistic Regression

Train/test split, Logistic Regression with `class_weight='balanced'` if imbalanced. Report Recall, F1, confusion matrix.

In [ ]:
# X_train, X_test, y_train, y_test = ...
# lr = LogisticRegression(class_weight='balanced', random_state=42)
# lr.fit(X_train, y_train)
# y_pred = lr.predict(X_test)
# print(classification_report(y_test, y_pred))
# print(confusion_matrix(y_test, y_pred))
# recall_score(y_test, y_pred, pos_label=1), f1_score(y_test, y_pred, pos_label=1)

---

## 6. Random Forest & tuning

Random Forest, then GridSearchCV. Compare to baseline.

In [ ]:
# rf = RandomForestClassifier(class_weight='balanced', random_state=42)
# Optional: GridSearchCV for n_estimators, max_depth, etc.
# rf.fit(X_train, y_train)
# Evaluate on X_test: Recall, F1, confusion matrix.

---

## 7. Evaluation and comparison

Comparison table: Recall, F1, ROC-AUC for baseline and improved model. Do not tune on test set.

In [ ]:
# Results table and optional ROC curve.

---

## 8. Feature importance

Plot feature importance (e.g. from Random Forest). Interpret for logistics manager.

In [ ]:
# Feature importance plot and short interpretation.